In [1]:
%load_ext autoreload
%autoreload 2

from matplotlib import pyplot as plt
import matplotlib.pyplot as plt
import numpy as np
from search_inputs import inputs, getSearchMap, download_location
import os
from ipyleaflet import Map, basemaps

## Timeframe and Polarizations
Select the (multiple) desired polarizations you want to download.

In [2]:
inputs

Box(children=(DatePicker(value=datetime.datetime(2025, 7, 1, 0, 0), description='Start Date:'), DatePicker(val…

In [3]:
m, searchState = getSearchMap(inputs, basemap=basemaps.Esri.WorldImagery)
m

Map(center=[37.5531, -109.6914], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', '…

["'type': 'REVERSE': 'report': Reversed polygon winding order"]


Finished Search with 18 results.


## Select a path
Once selected, the next cell will show you the footprints of the scenes that will actually be downloaded.

In [4]:
paths = searchState.pathSelector()
paths

Dropdown(description='Relative Orbit:', options=(('Path 1, ASCENDING (4)', 1), ('Path 73, ASCENDING (5)', 73),…

Updated Path
Updated Path


In [5]:
searchState.summaryMap(basemap=basemaps.Esri.WorldImagery)
# searchState.summaryMap()

Map(center=[37.5531, -109.6914], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', '…

## File Preparation and Download/Mosaic
The following cells aggregate scenes by date, and then use gdal's vsicurl prefixes to download cropped and mosaiced scenes of just your study area.

### First pick a download location

In [6]:
download_location

Text(value='/', description='Folder:', placeholder='//Enter/Valid/Folder/Paths')

In [7]:
polarizations = list(inputs.children[2].value)

In [8]:
dates = [scene.properties['startTime'].split('T')[0] for scene in searchState.scenes2download]
udates = np.unique(np.array(dates))
frames = [scene.properties['frameNumber'] for scene in searchState.scenes2download]
uframes = np.unique(np.array(frames))

download_folder = download_location.value
out_folders = [os.path.join(download_folder, f'./{pol}') for pol in polarizations]

for out_folder in out_folders:
    if not os.path.exists(out_folder):
        os.makedirs(out_folder)

target_extent = searchState.search_coordinates
latitudes = [coord[1] for coord in searchState.search_coordinates]
longitudes = [coord[0] for coord in searchState.search_coordinates]
te = f'{np.min(longitudes)} {np.min(latitudes)} {np.max(longitudes)} {np.max(latitudes)}'

In [9]:
bash_path = os.path.join(download_folder, './dl.sh')
cmds = []
n_proc = 8

for d in udates:
    files2download = [scene.properties['url'] for scene in searchState.scenes2download if d in scene.properties['startTime']]
    for pol, out_folder in zip(polarizations, out_folders):
        outpath = os.path.join(out_folder, f"./{d.replace('-', '')}_{pol}.tif")
        bandName = f'/science/LSAR/GSLC/grids/frequencyA/{pol}'
        files2download_pol = [f'NETCDF:/vsicurl/{file}:{bandName}' for file in files2download]
        gdal_command = f"gdalwarp -te {te} -te_srs EPSG:4326 \
    -srcnodata nan -dstnodata nan -r nearest -wo INIT_DEST=NO_DATA \
    -of GTiff -overwrite {' '.join(files2download_pol)} {outpath}"
        cmds.append(gdal_command)

bash_script = ''
for i in range(len(cmds)):
    bash_script += cmds[i]
    if (i+1) % n_proc == 0:
        bash_script += '\nwait\n'
    else:
        bash_script += ' &\n'
bash_script += 'wait'

with open(bash_path, "w") as dlsh:
    dlsh.write(bash_script)

## Run download script
The for-loop above generates the gdal commmands for downloading the GSLCS to disk so that they can be run either in this notebook or also in a screen while the terminal is closed. To run in a screen, 
```
screen -S dlnisar
bash <bash_path.sh>
```
Then press `ctr+a d` to allow the downloads to run in the background.

Or run the script directly from this notebook below.

In [10]:
!bash {bash_path}

Creating output file that is 7360P x 8115L.
Processing NETCDF:/vsicurl/https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GSLC_BETA_V1/NISAR_L2_PR_GSLC_003_155_D_054_7700_SHNA_A_20251027T225421_20251027T225457_X05009_N_P_J_001/NISAR_L2_PR_GSLC_003_155_D_054_7700_SHNA_A_20251027T225421_20251027T225457_X05009_N_P_J_001.h5:/science/LSAR/GSLC/grids/frequencyA/HH [1/1] : 0Creating output file that is 7360P x 8115L.
Processing NETCDF:/vsicurl/https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GSLC_BETA_V1/NISAR_L2_PR_GSLC_009_155_D_054_7700_SHNA_A_20260107T225424_20260107T225500_X05009_N_P_J_001/NISAR_L2_PR_GSLC_009_155_D_054_7700_SHNA_A_20260107T225424_20260107T225500_X05009_N_P_J_001.h5:/science/LSAR/GSLC/grids/frequencyA/HH [1/1] : 0Creating output file that is 7360P x 8115L.
Processing NETCDF:/vsicurl/https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GSLC_BETA_V1/NISAR_L2_PR_GSLC_010_155_D_054_7700_SHNA_A_20260119T225425_20260119T225501_X05010_N_P_J_001/NISAR_L2_PR_GSL

## Preview Data
The notebook `./example_analysis.ipynb` has a few examples of loading and forming interferograms from the downloaded SLCs.